# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Binary classification → ranking via calibrated probabilities.**

Lane 2 (Refresh / Content Opportunity Scoring) is framed as a **binary classification** task: *will this page decline in search impressions?* The classifier outputs calibrated probabilities (0–1), and we **rank** pages by those probabilities to produce a prioritized refresh queue.

Why classification, not clustering? Because we have a clear, observable outcome (decline vs. not-decline) and we want to predict it — clustering would group pages by similarity, but wouldn't tell us which groups need refresh action. The ranking emerges naturally from the probability estimates, making every position in the queue interpretable.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label = (trend_direction == "down")`

This is a **defined rule over an observed outcome**:
- `trend_direction` is derived from comparing impressions in the most recent 30-day window vs. the prior 30-day window
- A page is labeled "down" when it loses more than 20% of its impressions month-over-month
- This makes the label observable, reproducible, and threshold-based

**Critically:** Because the label is derived from `trend_direction`, neither `trend_direction` nor `trend_pct` can ever be a model feature — that would be direct label leakage.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@K** (specifically Precision@50)

Why Precision@K?
- Content teams work from the **top of the queue down** — they have limited editing resources
- Precision@50 answers: *"If I take the model's top-50 recommendations, how many are genuinely declining?"*
- A false positive costs real money (~$150–$500 per article refresh wasted on a page that doesn't need it)

**Supporting metrics:** ROC AUC (discrimination ability across all thresholds) and Average Precision (area under the precision-recall curve). We always report against the base rate (54.2% declining) so readers can judge if precision is meaningful.

**What 'good' means:** The baseline hand-rule achieves Precision@50 = 0.240. A model is useful if it significantly exceeds that — anything above 0.600 means the top-50 queue is more than 2.5× better than hand-rules at finding real decliners.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"One row = one content page")
print(f"Shape: {df.shape[0]:,} pages × {df.shape[1]} columns")
print(f"\nUnit of analysis: each row represents a single pseudonymized content page")
print(f"with its keyword context, content properties, 90-day search performance,")
print(f"engagement metrics, and trend direction.")
print(f"\nClients: {df['client_id'].nunique()} pseudonymized clients")
print(f"Declining: {(df['trend_direction'] == 'down').sum():,} pages ({(df['trend_direction'] == 'down').mean():.1%})")
df.head(3)

One row = one content page
Shape: 30,000 pages × 44 columns

Unit of analysis: each row represents a single pseudonymized content page
with its keyword context, content properties, 90-day search performance,
engagement metrics, and trend direction.

Clients: 32 pseudonymized clients
Declining: 16,262 pages (54.2%)


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule like "refresh everything older than 180 days" or "refresh pages with > 1,000 impressions and declining" ignores the **interaction** between signals:

1. **Content age alone isn't enough.** Some old pages are stable; some young pages are already declining. The relationship between age and decline depends on engagement quality and keyword dynamics.

2. **Signals interact non-linearly.** A page with high impressions but low CTR is different from a page with low impressions and low CTR — the first is a CTR-optimization opportunity, the second may be irrelevant. No single threshold captures this.

3. **Multiple feature dimensions matter simultaneously.** The top-4 features by importance — days_with_impressions, log_impressions_90d, avg_position, content_age_days — all contribute meaningfully, and their combined effect isn't additive.

4. **The baseline hand-rule proves the gap.** A carefully designed 4-component weighted rule (visibility + freshness risk + position opportunity + depth gap) only achieves Precision@50 = 0.240. The Random Forest reaches 0.740 — a ~3× lift — precisely because it captures the messy interactions that hand-rules miss.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.